In [21]:
import numpy as np
import pandas as pd

#set up for experimentation
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal/results/NIJ").resolve()

#path to hc edge stability
hc_path = project_root/"graphs_HC"/"NIJ_HC_edge_stability.csv"
dagslam_path = project_root/"graphs_DAGSLAM"/ "NIJ_DAGSLAM_edge_stability.csv"
dagbagm_path = project_root/"graphs_DAGBagM"/"NIJ_DAGBagM_edge_stability.csv"

df_hc = pd.read_csv(hc_path, index_col=0)
df_dagslam = pd.read_csv(dagslam_path, index_col=0)
df_dagbagm = pd.read_csv(dagbagm_path, index_col=0)

In [6]:
df_hc

,Gender_F,Race_BLACK,Age_at_Release_18-22,Age_at_Release_23-27,Age_at_Release_28-32,Age_at_Release_33-37,Age_at_Release_38-42,Age_at_Release_43-47,Age_at_Release_48 or older,Gang_Affiliated,...,Prior_Arrest_Episodes_Property_high,Prior_Arrest_Episodes_Drug_high,Prior_Conviction_Episodes_Felony_high,Percent_Days_Employed,Jobs_Per_Year,Avg_Days_per_DrugTest,DrugTests_Cocaine_Positive,Delinquency_Reports_high,Supervision_Risk_Score_First,Recidivism_Within_3years
Gender_F,0.00,0.95,0.95,0.30,0.0,0.00,0.00,0.00,0.0,0.35,...,0.05,0.00,0.00,0.00,0.00,0.50,0.00,0.70,0.00,0.85
Race_BLACK,0.05,0.00,0.45,0.60,0.0,0.00,0.00,0.00,0.0,0.35,...,0.35,0.10,0.00,1.00,0.80,1.00,1.00,0.05,0.00,0.05
Age_at_Release_18-22,0.00,0.45,0.00,0.95,1.0,1.00,1.00,1.00,1.0,0.70,...,0.00,0.45,1.00,0.00,0.00,0.10,0.25,0.00,0.20,0.35
Age_at_Release_23-27,0.00,0.30,0.05,0.00,1.0,1.00,1.00,1.00,1.0,0.25,...,0.05,0.60,0.90,0.00,0.00,0.65,0.80,0.00,0.20,0.40
Age_at_Release_28-32,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.10,...,0.05,0.00,0.00,0.00,0.00,0.00,0.65,0.00,0.20,0.00
Age_at_Release_33-37,0.00,0.00,0.00,0.00,1.0,0.00,0.95,0.70,1.0,0.00,...,0.00,0.05,0.00,0.00,0.00,0.00,0.25,0.00,0.80,0.00
Age_at_Release_38-42,0.00,0.00,0.00,0.00,1.0,0.05,0.00,0.65,1.0,0.00,...,0.00,0.00,0.00,0.00,0.00,0.05,0.15,0.00,0.80,0.00
Age_at_Release_43-47,0.00,0.00,0.00,0.00,1.0,0.30,0.35,0.00,1.0,0.05,...,0.90,0.00,0.65,0.00,0.00,0.00,0.05,0.00,0.80,0.10
Age_at_Release_48 or older,0.00,0.10,0.00,0.00,1.0,0.00,0.00,0.00,0.0,0.25,...,0.90,0.00,0.00,0.05,1.00,0.00,0.15,0.00,1.00,0.20
Gang_Affiliated,0.00,0.15,0.30,0.75,0.0,0.05,0.00,0.00,0.0,0.00,...,0.00,0.00,0.05,0.00,0.00,0.70,0.20,0.90,0.00,0.90


In [13]:
TARGET = "Recidivism_Within_3years"
#stability threshold
tau = 0.5

def calculate_stable_parents(tau,df):
    parents_mask = df[TARGET] >= tau
    stable_parents = list(df.index[parents_mask])
    n_stable_parents = len(stable_parents)
    print("Stable parents (freq >=", tau , stable_parents)
    print("Number of stable parents:", n_stable_parents)

calculate_stable_parents(0.5,df_hc)
calculate_stable_parents(0.8,df_hc)

Stable parents (freq >= 0.5 ['Gender_F', 'Gang_Affiliated']
Number of stable parents: 2
Stable parents (freq >= 0.8 ['Gender_F', 'Gang_Affiliated']
Number of stable parents: 2


In [17]:
def compute_stable_edges(df):
    # maximum possible directed edges
    d = df.shape[0]
    n_possible = d * (d - 1)
    
    # count distinct edges
    distinct_edges = np.sum(df.to_numpy() > 0)
    print("Distinct edges seen in any run:", distinct_edges)
    
    n_ge_05 = np.sum(df.to_numpy() >= 0.5)
    n_ge_08 = np.sum(df.to_numpy() >= 0.8)
    
    prop_05_of_distinct = n_ge_05 / distinct_edges if distinct_edges else 0
    prop_08_of_distinct = n_ge_08 / distinct_edges if distinct_edges else 0
    print("Edges with freq >= 0.5:", n_ge_05, prop_05_of_distinct*100)
    print("Edges with freq >= 0.8:", n_ge_08, prop_08_of_distinct*100)
    
    prop_05_of_possible = n_ge_05 / n_possible
    prop_08_of_possible = n_ge_08 / n_possible


In [18]:
compute_stable_edges(df_hc)

Distinct edges seen in any run: 153
Edges with freq >= 0.5: 75 49.01960784313725
Edges with freq >= 0.8: 56 36.60130718954248


In [20]:
calculate_stable_parents(0.5,df_dagslam)
calculate_stable_parents(0.8,df_dagslam)
compute_stable_edges(df_dagslam)

Stable parents (freq >= 0.5 ['Prior_Arrest_Episodes_Felony_high', 'Percent_Days_Employed']
Number of stable parents: 2
Stable parents (freq >= 0.8 ['Percent_Days_Employed']
Number of stable parents: 1
Distinct edges seen in any run: 117
Edges with freq >= 0.5: 88 75.21367521367522
Edges with freq >= 0.8: 72 61.53846153846154


In [23]:
calculate_stable_parents(0.5,df_dagbagm)
calculate_stable_parents(0.8,df_dagbagm)
compute_stable_edges(df_dagbagm)

Stable parents (freq >= 0.5 ['Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Conviction_Episodes_Felony_high', 'Jobs_Per_Year', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First']
Number of stable parents: 12
Stable parents (freq >= 0.8 ['Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_48 or older', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Conviction_Episodes_Felony_high', 'Supervision_Risk_Score_First']
Number of stable parents: 7
Distinct edges seen in any run: 298
Edges with freq >= 0.5: 128 42.95302013422819
Edges with freq >= 0.8: 80 26.845637583892618
